# Power BI ideas in a browser: slicers, cards and listings

**Time:** 45 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Use all three scenario tabs; audit filter propagation; build a small Python widget dashboard and compare it with the full browser dashboard.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/05_dashboard_lab.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [1]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

Ready: public poster aggregates + clearly labeled fictional facility fixtures.


## A dashboard is a set of coordinated questions

The original report contains maps, country/city/PEPFAR slicers, decade classifications, tables, treemaps and cards. This browser implementation retains the scenario tabs, geographic slicers, year animation, line listings and aggregates. The public layer works at the poster's published reporting units; the fictional layer demonstrates facility-level inspection and service filters.

**Filter contract:** scenario and year define the current slice. Country and city select a cohort. Public scope chooses exactly one of city, district or whole portfolio. A fictional threshold changes the numerator; it does not remove facilities from the denominator. Listing search is deliberately local to the table and its label says so. Time-series charts retain the cohort while showing all years and all three scenarios.

In [2]:
dashboard(scenario='RCP 4.5')

Open full-screen dashboard

## Build the underlying calculation yourself

The next cell uses `ipywidgets` in JupyterLite. It needs the Python kernel, while the full dashboard above uses browser JavaScript and works without a kernel. The snapshot below is executed in the static book; use the live notebook to change Python widgets.

In [3]:
import ipywidgets as widgets
scenario_widget=widgets.Dropdown(options=SCENARIOS,description='Scenario:')
year_widget=widgets.SelectionSlider(options=YEARS,value=2030,description='Year:')
country_widget=widgets.Dropdown(options=['All']+sorted(facilities.country.unique()),description='Country:')
threshold_widget=widgets.FloatSlider(min=.01,max=.99,step=.01,value=.1,description='Threshold:')
def update(scenario,year,country,threshold):
    rows=select(facilities,exposure,scenario= scenario,year=year,country=country)
    result=cards(rows,threshold)
    display(pd.DataFrame([result]).round(2))
    display(rows[['facility_id','city','annual_probability']].head(5))
output=widgets.interactive_output(update,{'scenario':scenario_widget,'year':year_widget,'country':country_widget,'threshold':threshold_widget})
display(widgets.VBox([scenario_widget,year_widget,country_widget,threshold_widget,output]))

In [4]:
# Independently reconcile the card's numerator with its row-level predicate.
rows=select(facilities,exposure,scenario='RCP 4.5',year=2050,country='Thailand')
manual_count=int((rows.annual_probability>=.1).sum())
assert cards(rows)['above_threshold']==manual_count
print('Reconciled:',manual_count,'fictional exposed facilities of',len(rows))

Reconciled: 18 fictional exposed facilities of 48


## Lab: audit, do not just explore

1. In the public dashboard, verify portfolio cards for 2030: 108, 107 and 108 for RCP 2.6, 4.5 and 8.5.
2. Move to 2100 and verify 321, 331 and 412.
3. Choose Published cities → Thailand → Bangkok and reconcile the listing with the cards.
4. Reset filters. Switch to the fictional layer and set the same cohort as the Python example.
5. Export a CSV. Count unique IDs and confirm that each ID appears once. Explain why the service-load card is a workload proxy, not a patient count.
6. Raise the threshold. The exposed count should stay the same or decrease; the cohort denominator should stay constant.

**Deliverable:** A screenshot with a short caption naming all active filters, plus an exported CSV and a three-line reconciliation calculation. The static site includes keyboard-accessible controls and a numeric table so colour is never the only way to read a result.